In [ ]:
import mne
from mne.datasets import eegbci

subject = 1
runs = [6]

raw_fnames = eegbci.load_data(subject, runs)
raw = mne.io.read_raw_edf(raw_fnames[0], preload=True)

raw

In [ ]:
print(raw)
print(raw.info["sfreq"])
print(len(raw.ch_names))
print(raw.ch_names[:10])
raw.info.keys()

In [ ]:
raw.plot(n_channels=10, duration=5)

In [ ]:
events, event_id = mne.events_from_annotations(raw)

print(events[:10])
print(event_id)

In [ ]:
picks = mne.pick_types(raw.info, eeg=True, exclude="bads")

epochs = mne.Epochs(
    raw,
    events,
    event_id=event_id,
    tmin=-1.0,
    tmax=4.0,
    picks=picks,
    baseline=None,
    preload=True
)

epochs

In [ ]:
print(event_id)
print(epochs)
print(epochs.get_data().shape)
epochs[event_id.keys().__iter__().__next__()]

In [ ]:
label_meaning = {
    "T0": "rest",
    "T1": "both fists",
    "T2": "both feet"
}

print("event_id:", event_id)
print("label meaning:", label_meaning)

In [ ]:
import numpy as np
import pandas as pd

data = epochs.get_data()
labels_int = epochs.events[:, -1]

inv_event_id = {v: k for k, v in event_id.items()}
labels_name = np.array([inv_event_id[x] for x in labels_int])

print("data shape:", data.shape)
print("first 10 labels:", labels_name[:10])

In [ ]:
label_counts = pd.Series(labels_name).value_counts().sort_index()
print(label_counts)

In [ ]:
print("single epoch shape:", data[0].shape)
print("time points:", len(epochs.times))
print("time range:", epochs.times[0], "to", epochs.times[-1])

In [ ]:
import matplotlib.pyplot as plt

sample_idx = 0
channels_to_plot = [0, 1, 2, 3, 4]

plt.figure(figsize=(12, 6))
for ch in channels_to_plot:
    plt.plot(epochs.times, data[sample_idx, ch, :], label=epochs.ch_names[ch])

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"Single Epoch Example - Label: {labels_name[sample_idx]}")
plt.legend()
plt.show()

In [ ]:
example_indices = {}
for label in ["T0", "T1", "T2"]:
    idx = np.where(labels_name == label)[0][0]
    example_indices[label] = idx

channels_to_plot = [0, 1, 2, 3, 4]

for label, idx in example_indices.items():
    plt.figure(figsize=(12, 4))
    for ch in channels_to_plot:
        plt.plot(epochs.times, data[idx, ch, :], label=epochs.ch_names[ch])
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title(f"Example Epoch - {label} ({label_meaning[label]})")
    plt.legend()
    plt.show()

In [ ]:
avg_by_label = {}
for label in ["T0", "T1", "T2"]:
    avg_by_label[label] = data[labels_name == label].mean(axis=0)  # shape: (channels, timepoints)

channels_to_plot = [0, 1, 2, 3, 4]
for ch in channels_to_plot:
    plt.figure(figsize=(12, 4))
    for label in ["T0", "T1", "T2"]:
        plt.plot(epochs.times, avg_by_label[label][ch], label=f"{label} ({label_meaning[label]})")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title(f"Average Waveform by Label - Channel: {epochs.ch_names[ch]}")
    plt.legend()
    plt.show()

In [ ]:
epoch_std = data.std(axis=(1, 2))
print("epoch std summary:")
print(pd.Series(epoch_std).describe())

plt.figure(figsize=(10, 4))
plt.plot(epoch_std)
plt.xlabel("Epoch Index")
plt.ylabel("Std Across Channels and Time")
plt.title("Epoch-Level Variability Check")
plt.show()


In [ ]:
max_idx = epoch_std.argmax()
min_idx = epoch_std.argmin()

print("max std epoch index:", max_idx, "value:", epoch_std[max_idx])
print("min std epoch index:", min_idx, "value:", epoch_std[min_idx])

channels_to_plot = [0, 1, 2, 3, 4]
for ch in channels_to_plot:
    plt.figure(figsize=(12, 4))

    plt.plot(epochs.times, data[max_idx][ch], label=f"epoch {max_idx} (max std)")
    plt.plot(epochs.times, data[min_idx][ch], label=f"epoch {min_idx} (min std)")

    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title(f"Max vs Min Std Epoch Comparison - Channel: {epochs.ch_names[ch]}")
    plt.legend()
    plt.show()